<a href="https://colab.research.google.com/github/AdithyaWijerathne02/Sri-Lanka-Economy-Dashboard/blob/main/Sri_Lanka_Economic_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas requests plotly streamlit sqlite-utils

In [ ]:
import os

os.makedirs("data_raw", exist_ok=True)
os.makedirs("data_clean", exist_ok=True)

print("Folders created!")

Folders created!


In [ ]:
import requests
import pandas as pd

def wb_api(url):
    return pd.json_normalize(requests.get(url).json()[1])

# GDP Growth
df_gdp = wb_api("https://api.worldbank.org/v2/country/LKA/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=60")
df_gdp.to_csv("data_raw/gdp.csv", index=False)

# Inflation
df_inf = wb_api("https://api.worldbank.org/v2/country/LKA/indicator/FP.CPI.TOTL.ZG?format=json&per_page=60")
df_inf.to_csv("data_raw/inflation.csv", index=False)

# Unemployment
df_unemp = wb_api("https://api.worldbank.org/v2/country/LKA/indicator/SL.UEM.TOTL.ZS?format=json&per_page=60")
df_unemp.to_csv("data_raw/unemployment.csv", index=False)

# Exchange Rate
df_ex = wb_api("https://api.worldbank.org/v2/country/LKA/indicator/PA.NUS.FCRF?format=json&per_page=60")
df_ex.to_csv("data_raw/exchange_rate.csv", index=False)

print("All raw datasets downloaded!")

All raw datasets downloaded!


In [ ]:
# Clean each dataset
df_gdp_clean = df_gdp[['date','value']].rename(columns={'date':'year','value':'gdp_growth'}).dropna()
df_unemp_clean = df_unemp[['date','value']].rename(columns={'date':'year','value':'unemployment'}).dropna()
df_inf_clean = df_inf[['date','value']].rename(columns={'date':'year','value':'inflation'}).dropna()
df_ex_clean = df_ex[['date','value']].rename(columns={'date':'year','value':'exchange_rate'}).dropna()

# Save
df_gdp_clean.to_csv("data_clean/gdp_clean.csv", index=False)
df_unemp_clean.to_csv("data_clean/unemployment_clean.csv", index=False)
df_inf_clean.to_csv("data_clean/inflation_clean.csv", index=False)
df_ex_clean.to_csv("data_clean/exchange_rate_clean.csv", index=False)

print("Clean data saved!")

Clean data saved!


In [ ]:
df_final = (
    df_gdp_clean
    .merge(df_unemp_clean, on="year")
    .merge(df_inf_clean, on="year")
    .merge(df_ex_clean, on="year")
)

df_final.to_csv("data_clean/final_economic_data.csv", index=False)
df_final

,year,gdp_growth,unemployment,inflation,exchange_rate
0,2023,-2.329848,4.528,16.541174,327.506533
1,2022,-7.349193,4.528,49.721102,322.632674
2,2021,4.207476,4.981,7.014781,198.764317
3,2020,-4.624515,5.364,6.153945,185.592558
4,2019,-0.220484,4.670,3.528394,178.744925
5,2018,2.310084,4.318,2.135038,162.464859
6,2017,6.460681,4.046,7.704138,152.446414
7,2016,5.053625,4.242,3.958888,145.581667
8,2015,4.205955,4.519,3.768368,135.856913
9,2014,6.377979,4.157,3.179002,130.564685


In [ ]:
import sqlite3

conn = sqlite3.connect("sri_lanka_economy.db")

df_final.to_sql("economic_data", conn, if_exists="replace", index=False)

print("SQL Database created and data loaded!")

SQL Database created and data loaded!


In [ ]:
query = """
SELECT * FROM economic_data
WHERE year > 2000
ORDER BY year;
"""

pd.read_sql_query(query, conn)

,year,gdp_growth,unemployment,inflation,exchange_rate
0,2001,-1.545408,7.900,14.158456,89.383013
1,2002,3.964676,8.760,9.551032,95.662065
2,2003,5.940269,8.220,6.314638,96.520951
3,2004,5.445061,8.380,7.575926,101.194457
4,2005,6.241748,7.670,11.639686,100.498052
5,2006,7.668292,6.500,10.020184,103.914446
6,2007,6.796826,5.970,15.842111,110.623233
7,2008,5.950088,5.220,22.564496,108.333763
8,2009,3.538912,5.850,3.464963,114.944783
9,2010,8.015967,4.784,6.217649,113.064480


In [ ]:
pd.read_sql_query("""
SELECT
    MIN(gdp_growth) AS min_gdp,
    MAX(inflation) AS max_inflation,
    AVG(unemployment) AS avg_unemployment
FROM economic_data;
""", conn)

,min_gdp,max_inflation,avg_unemployment
0,-7.349193,49.721102,7.349455


In [ ]:
from google.colab import files
files.download("data_clean/final_economic_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%writefile dashboard_app.py
import streamlit as st
import pandas as pd
import plotly.express as px

df = pd.read_csv("data_clean/final_economic_data.csv")

st.title("🇱🇰 Sri Lanka Economic Dashboard")

st.subheader("GDP Growth (%)")
st.plotly_chart(px.line(df, x="year", y="gdp_growth"))

st.subheader("Inflation Rate (%)")
st.plotly_chart(px.line(df, x="year", y="inflation"))

st.subheader("Unemployment (%)")
st.plotly_chart(px.line(df, x="year", y="unemployment"))

st.subheader("Exchange Rate (LKR per USD)")
st.plotly_chart(px.line(df, x="year", y="exchange_rate"))

Writing dashboard_app.py


In [ ]:
!streamlit run dashboard_app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-05-13 12:40:13.341 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.176.34:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://whole-nails-drop.loca.lt
  Stopping...
^C


In [ ]:
!pkill streamlit

In [ ]:
!streamlit run dashboard_app.py --server.port 8501 --server.enableCORS false

2026-05-13 13:10:21.857 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            


2026-05-13 13:10:23.141 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.176.34:8501

